# M22 Discovery Entrypoint — Telco Dataset Authoring Notebook

Telco Customer Churn authoring entrypoint for Atlas DataFlow (Project Spec S0011/S0012/S0013).

This notebook is the Telco-specific authoring surface for
`data/raw/telco-customer-churn.csv`. It loads the raw CSV, verifies known
structural facts, records authoring observations, and assembles a narrow
`dataset_modeling_intent.v1` authoring contract (Project Spec S0013) from
those observations. Common loading, inspection, and modeling-intent-building
logic is implemented once as reusable, tested helper functions in
`pipeline/discovery_evidence.py` (Project Specs S0012/S0013); this notebook
calls those helpers and keeps only Telco-specific decisions (expected
structural facts, target column, identifier columns, feature review notes)
visible inline.

## Boundaries

- This notebook is an **authoring notebook only**.
- Notebook output is **not** the final operational source of truth.
- This notebook does not train models, select model families, create release
  candidates, validate publisher candidates, promote releases, mutate
  registry state, or change API/UI behavior.
- Any local output produced by this notebook is a **non-promoted authoring
  artifact**. It must not be treated as an official contract, release
  candidate, publisher run, registry file, model binary, or UI data fixture.
- Downstream contract derivation, model training, and publication are
  separate, later stages driven by separate, explicitly authorized specs.

## Usage

Run locally with the default repository-relative parameters, or override
`repo_root` explicitly (for example under papermill) if the notebook is
executed from outside the repository working directory:

```
papermill notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb output.ipynb \
    -p repo_root /path/to/atlas-dataflow
```

Do not rely on implicit notebook state or hidden local paths as inputs to
later pipeline stages.

In [1]:
# Side-effect boundary — these operations are explicitly forbidden in this
# authoring entrypoint. This is checked before any dataset loading occurs.
FORBIDDEN_SIDE_EFFECTS = {
    "model_training": False,
    "model_family_selection": False,
    "release_candidate_creation": False,
    "publisher_run_creation": False,
    "release_promotion": False,
    "registry_state_mutation": False,
    "api_behavior_change": False,
    "ui_behavior_change": False,
    "reusable_helper_module_creation": False,
    "notebook_output_committed_as_official_artifact": False,
    "modeling_intent_treated_as_execution_contract": False,
}
assert all(not v for v in FORBIDDEN_SIDE_EFFECTS.values()), (
    "Side-effect boundary violated; this entrypoint must not perform any "
    "forbidden operation."
)
print("Side-effect boundaries confirmed:", FORBIDDEN_SIDE_EFFECTS)

Side-effect boundaries confirmed: {'model_training': False, 'model_family_selection': False, 'release_candidate_creation': False, 'publisher_run_creation': False, 'release_promotion': False, 'registry_state_mutation': False, 'api_behavior_change': False, 'ui_behavior_change': False, 'reusable_helper_module_creation': False, 'notebook_output_committed_as_official_artifact': False}


## Parameters and repository-relative dataset path

`dataset_relative_path` is repository-relative and explicit — no implicit or
hidden absolute paths. `repo_root` defaults to the current working directory
(the expected convention when running this notebook from the repository
root) and may be overridden explicitly, for example by papermill, when the
notebook is executed from elsewhere.

In [2]:
# Parameters — supply explicit overrides here or via papermill; the defaults
# below are the Telco authoring defaults for this notebook.
dataset_slug = "telco-customer-churn"          # Fixed authoring identity for this notebook.
dataset_relative_path = "data/raw/telco-customer-churn.csv"  # Repository-relative, explicit.
repo_root = None                                # Optional override (str); None = current working directory.
target_column = "Churn"                         # Observed target column for this dataset.

## Raw CSV loading

Resolve the repository-relative dataset path against `repo_root` (or the
current working directory when `repo_root` is not supplied) and load the raw
CSV with the standard library `csv` module only. This notebook intentionally
avoids new repository dependencies (for example `pandas`) — none are
authorized by this spec.

In [3]:
import sys
import json
from pathlib import Path

from pipeline.discovery_evidence import (
    resolve_repository_path,
    load_dataset_csv,
    summarize_structure,
    observe_authoring_fields,
    summarize_target_column,
    summarize_identifier_columns,
    derive_feature_candidates,
    authoring_helper_evidence_policy,
    build_dataset_modeling_intent,
)

_repo_root_path = Path(repo_root) if repo_root else Path.cwd()
if str(_repo_root_path) not in sys.path:
    sys.path.insert(0, str(_repo_root_path))

dataset_path = resolve_repository_path(dataset_relative_path, repo_root=repo_root)

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {dataset_path}. "
        f"Expected the official raw CSV at repository-relative path "
        f"'{dataset_relative_path}' under repo_root '{_repo_root_path}'. "
        "Supply an explicit repo_root override if running outside the "
        "repository working directory."
    )

raw_rows = load_dataset_csv(dataset_path)
structure = summarize_structure(raw_rows)
raw_columns = structure["ordered_columns"]

print(f"dataset_slug   : {dataset_slug}")
print(f"dataset_path   : {dataset_path}")
print(f"row_count      : {structure['row_count']}")
print(f"column_count   : {structure['column_count']}")

ModuleNotFoundError: No module named 'pipeline'

## Structural verification

Verify the row count, column count, ordered column list, and per-field
authoring observations (inferred type, blank string count, null-like count,
cardinality, reduced sample bounds) via the reusable
`observe_authoring_fields` helper from `pipeline/discovery_evidence.py`.
These are structural facts recorded by this Project Spec (S0011/S0012)
against the committed source file. A mismatch means the source CSV changed
since this notebook's observations were authored and must be reviewed
explicitly before trusting the rest of this notebook's recorded
observations.

In [ ]:
EXPECTED_ROW_COUNT = 7043
EXPECTED_COLUMN_COUNT = 21
EXPECTED_COLUMNS = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents",
    "tenure", "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling",
    "PaymentMethod", "MonthlyCharges", "TotalCharges", "Churn",
]

assert structure["row_count"] == EXPECTED_ROW_COUNT, (
    f"Row count {structure['row_count']} does not match the recorded authoring "
    f"observation ({EXPECTED_ROW_COUNT}). The source CSV may have changed; "
    "revisit this notebook's recorded structural observations before "
    "trusting downstream sections."
)
assert structure["column_count"] == EXPECTED_COLUMN_COUNT, (
    f"Column count {structure['column_count']} does not match the recorded "
    f"authoring observation ({EXPECTED_COLUMN_COUNT})."
)
assert raw_columns == EXPECTED_COLUMNS, (
    "Ordered column list does not match the recorded authoring observation. "
    f"Observed: {raw_columns}"
)

field_observations = observe_authoring_fields(raw_rows, raw_columns)

print("Structural verification passed: row_count, column_count, ordered columns match.")
print("Field observations (inferred_type, blank_string_count, null_like_count, cardinality):")
print(json.dumps(
    [
        {k: v for k, v in obs.items() if k != "reduced_sample_values"}
        for obs in field_observations
    ],
    indent=2,
))

## Target-column inspection — `Churn`

Inspect the observed target column labels and distribution via the reusable
`summarize_target_column` helper. `Churn` is the binary target column for
this dataset. The likely positive-class candidate for later modeling review
is `Yes`, but the final positive-label decision must be externalized by a
later modeling-intent contract spec — this notebook only records the
observation, it does not decide it.

In [ ]:
EXPECTED_TARGET_LABELS = {"No", "Yes"}
EXPECTED_TARGET_DISTRIBUTION = {"No": 5174, "Yes": 1869}
LIKELY_POSITIVE_CLASS_CANDIDATE = "Yes"  # Observation only — not a final decision.

target_summary = summarize_target_column(raw_rows, target_column)
target_label_set = set(target_summary["observed_labels"])
target_distribution = target_summary["observed_distribution"]

assert target_label_set == EXPECTED_TARGET_LABELS, (
    f"Observed target labels {sorted(target_label_set)} do not match the "
    f"recorded authoring observation {sorted(EXPECTED_TARGET_LABELS)}."
)
assert target_distribution == EXPECTED_TARGET_DISTRIBUTION, (
    f"Observed target distribution {target_distribution} does not match the "
    f"recorded authoring observation {EXPECTED_TARGET_DISTRIBUTION}. The "
    "source CSV may have changed; revisit this notebook's recorded "
    "observations before trusting downstream sections."
)
assert target_summary["is_authoritative"] is False, (
    "summarize_target_column must always report is_authoritative: False."
)

print(f"target_column                  : {target_column}")
print(f"observed_target_labels         : {sorted(target_label_set)}")
print(f"observed_target_distribution   : {target_distribution}")
print(f"likely_positive_class_candidate: {LIKELY_POSITIVE_CLASS_CANDIDATE} (observation only, not a final decision)")

## Identifier-column inspection — `customerID`

`customerID` is a per-row identifier candidate, not a modeling feature
candidate, inspected via the reusable `summarize_identifier_columns` helper.
It must be excluded from initial feature candidates without an explicit
override recorded elsewhere.

In [ ]:
IDENTIFIER_COLUMNS = ["customerID"]

identifier_summaries = summarize_identifier_columns(raw_rows, IDENTIFIER_COLUMNS)
customer_id_summary = identifier_summaries[0]

assert customer_id_summary["is_unique_per_row"], (
    f"Expected 'customerID' to be unique per row ({customer_id_summary['row_count']} rows), "
    f"observed {customer_id_summary['unique_count']} unique values. An identifier "
    "candidate that is not unique per row must be reviewed explicitly "
    "before being treated as an identifier."
)

print(f"identifier_columns       : {IDENTIFIER_COLUMNS}")
print(f"customerID_unique_count  : {customer_id_summary['unique_count']} (of {customer_id_summary['row_count']} rows)")
print("customerID is recorded as an identifier candidate and is excluded from "
      "initial feature candidates without explicit override.")

## Missing and blank-value inspection — `TotalCharges`

Surface the `TotalCharges` blank-string condition explicitly rather than
silently coercing it to numeric. Blank-value handling for `TotalCharges`
must be decided explicitly by a later modeling-intent contract spec before
final modeling intent — this notebook only records the observation.

In [ ]:
total_charges_obs = next(o for o in field_observations if o["name"] == "TotalCharges")
total_charges_blank_count = total_charges_obs["blank_string_count"]
total_charges_blank_tenure_values = sorted(
    {row["tenure"] for row in raw_rows if row["TotalCharges"].strip() == ""}
)

if total_charges_blank_count == 0:
    print(
        "No blank 'TotalCharges' values observed in this run of the CSV. "
        "This differs from this notebook's recorded authoring observation "
        "(11 blank values, all at tenure == '0') — revisit before trusting "
        "downstream authoring notes."
    )
else:
    print(f"total_charges_blank_count           : {total_charges_blank_count}")
    print(f"total_charges_blank_tenure_values   : {total_charges_blank_tenure_values}")

print(
    "'TotalCharges' must not be silently treated as fully numeric. Blank-value "
    "handling (for example impute-as-zero, drop, or a distinct missing-value "
    "indicator) must be decided explicitly by a later modeling-intent "
    "contract spec, not implicitly by this authoring notebook or by "
    "downstream training code."
)

## Feature-candidate overview

List non-target, non-identifier columns as initial feature candidates only,
via the reusable `derive_feature_candidates` helper. `SeniorCitizen` is
recorded separately as a binary numeric indicator, since it is already
represented as `0`/`1` in the raw CSV rather than as a raw categorical label.

In [ ]:
feature_candidate_columns = derive_feature_candidates(
    raw_columns, target_column=target_column, identifier_columns=IDENTIFIER_COLUMNS
)

senior_citizen_values = sorted({row["SeniorCitizen"] for row in raw_rows})
assert set(senior_citizen_values) == {"0", "1"}, (
    f"Expected 'SeniorCitizen' to be a binary numeric indicator with "
    f"observed values {{'0', '1'}}, observed {senior_citizen_values}."
)

excluded_from_feature_candidates = sorted(set(IDENTIFIER_COLUMNS) | {target_column})
print(f"excluded_from_feature_candidates : {excluded_from_feature_candidates}")
print(f"feature_candidate_columns        : {feature_candidate_columns}")
print(f"SeniorCitizen_observed_values    : {senior_citizen_values} (binary numeric indicator)")
print(
    "These are initial feature candidates only — final feature selection, "
    "encoding, and missing-value policy are decided by a later "
    "modeling-intent contract spec, not by this authoring notebook."
)

## Preliminary authoring observations

Assemble the authoring observations recorded by this notebook, plus the
reusable helper's reduced evidence policy confirmation
(`authoring_helper_evidence_policy`), into a single structure. This is an
authoring-time observation record intended to feed a later, separately
authorized modeling-intent contract spec — it is not itself an execution
contract, release candidate, or any other governed pipeline artifact.

In [ ]:
authoring_observations = {
    "dataset_slug": dataset_slug,
    "dataset_relative_path": dataset_relative_path,
    "row_count": structure["row_count"],
    "column_count": structure["column_count"],
    "ordered_columns": structure["ordered_columns"],
    "field_observations": field_observations,
    "target_column": target_column,
    "observed_target_labels": sorted(target_label_set),
    "observed_target_distribution": target_distribution,
    "likely_positive_class_candidate": LIKELY_POSITIVE_CLASS_CANDIDATE,
    "positive_class_decision_finalized": False,
    "identifier_columns": IDENTIFIER_COLUMNS,
    "total_charges_blank_count": total_charges_blank_count,
    "total_charges_blank_value_policy_decided": False,
    "feature_candidate_columns": feature_candidate_columns,
    "senior_citizen_observed_values": senior_citizen_values,
    "reduced_evidence_policy": authoring_helper_evidence_policy(),
    "notebook_boundary": "authoring_notebook_only",
    "notebook_output_is_not_final_operational_truth": True,
    "requires_later_modeling_intent_contract_spec": True,
}

print("Authoring observations:")
print(json.dumps(authoring_observations, indent=2))

## Dataset modeling intent — `dataset_modeling_intent.v1` (Project Spec S0013)

Assemble the reviewed authoring decisions above into a narrow
`dataset_modeling_intent.v1` object via the reusable
`build_dataset_modeling_intent` helper from `pipeline/discovery_evidence.py`.
This is an authoring-intent contract only — it is not an execution contract,
runtime contract, public contract, release candidate input, publisher input,
registry artifact, API fixture, or UI fixture, and it is kept separate from
all of those. `TotalCharges` is recorded as a preparation review item pending
explicit blank-value handling, and `SeniorCitizen` is recorded as requiring
explicit semantic type intent, since its raw representation is numeric but
its semantic domain is binary. No model training, release assembly,
publication, registry mutation, API change, UI change, or notebook execution
is authorized by this section.

In [ ]:
MODELING_INTENT_FEATURE_REVIEW_NOTES = {
    "TotalCharges": (
        "Raw blank string values observed; requires explicit blank-value "
        "handling before execution-contract projection."
    ),
    "SeniorCitizen": (
        "Raw representation is numeric (0/1) but the semantic domain is "
        "binary; requires explicit type intent before encoding."
    ),
}
MODELING_INTENT_FEATURE_TYPE_OVERRIDES = {
    "SeniorCitizen": "requires_review",
}
MODELING_INTENT_BLANK_VALUE_POLICY_CANDIDATES = {
    "TotalCharges": "unresolved_pending_review",
}
MODELING_INTENT_OPEN_QUESTIONS = [
    "Final 'TotalCharges' blank-value handling policy is not yet decided "
    "(candidates include impute-as-zero, drop, or a distinct missing-value "
    "indicator).",
    "Final 'SeniorCitizen' semantic type/encoding treatment is not yet "
    "decided.",
    "Final positive-label decision for 'Churn' requires explicit "
    "confirmation despite the observed candidate recorded here.",
]

dataset_modeling_intent = build_dataset_modeling_intent(
    dataset_slug=dataset_slug,
    dataset_source_ref=dataset_relative_path,
    authoring_notebook_ref="notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb",
    columns=raw_columns,
    target_column=target_column,
    task_type="binary_classification",
    observed_labels=sorted(target_label_set),
    positive_label_candidate=LIKELY_POSITIVE_CLASS_CANDIDATE,
    observed_target_distribution=target_distribution,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_review_notes=MODELING_INTENT_FEATURE_REVIEW_NOTES,
    feature_type_intent_overrides=MODELING_INTENT_FEATURE_TYPE_OVERRIDES,
    blank_value_policy_candidates=MODELING_INTENT_BLANK_VALUE_POLICY_CANDIDATES,
    open_questions=MODELING_INTENT_OPEN_QUESTIONS,
)

assert dataset_modeling_intent["contract_version"] == "dataset_modeling_intent.v1"
assert dataset_modeling_intent["target_intent"]["is_final_training_configuration"] is False
assert not any(dataset_modeling_intent["modeling_intent_boundary_confirmations"].values()), (
    "dataset_modeling_intent must not be treated as an execution/runtime/public "
    "contract, release candidate input, publisher input, registry artifact, "
    "API fixture, or UI fixture."
)

print("Dataset modeling intent (draft authoring evidence — not an execution contract):")
print(json.dumps(dataset_modeling_intent, indent=2))

## Non-promoted local output summary

This notebook may optionally write `authoring_observations` and
`dataset_modeling_intent` to local files for the operator's own convenience
during authoring. Any such file is a **non-promoted, draft authoring
artifact** — it lives under `data/`, which is excluded from version control
by this repository's `.gitignore`, and it must not be treated as an official
contract, execution contract, release candidate, publisher run, registry
file, or UI data fixture. Promoting any of these observations into an
official artifact requires a separate, explicitly authorized implementation
request.

In [ ]:
write_local_output = False  # Set True locally to write the non-promoted summary files below.

local_output_dir = _repo_root_path / "data" / "raw" / "telco-customer-churn-authoring-output"
local_output_path = local_output_dir / f"{dataset_slug}-authoring-observations.json"
local_modeling_intent_output_path = local_output_dir / f"{dataset_slug}-modeling-intent.draft.json"

if write_local_output:
    local_output_dir.mkdir(parents=True, exist_ok=True)
    local_output_path.write_text(
        json.dumps(authoring_observations, indent=2), encoding="utf-8"
    )
    local_modeling_intent_output_path.write_text(
        json.dumps(dataset_modeling_intent, indent=2), encoding="utf-8"
    )
    print(f"Non-promoted authoring output written to: {local_output_path}")
    print(
        "Non-promoted, draft dataset_modeling_intent output written to: "
        f"{local_modeling_intent_output_path}"
    )
else:
    print(
        "write_local_output is False; no local output files were written. "
        f"If enabled, the non-promoted authoring output would be written to: "
        f"{local_output_path}\n"
        "and the non-promoted, draft modeling-intent output would be written to: "
        f"{local_modeling_intent_output_path}"
    )

print(
    "These output paths are local and non-promoted. Neither is an official "
    "contract, execution contract, release candidate, publisher run, "
    "registry file, or UI data fixture. Promotion requires a separate, "
    "explicitly authorized implementation request."
)